# VORQ · 02 — Engine Core

Contains: ExposureGraph · EventExtractor · RiskScoringEngine · ExplainabilityEngine · SimulationOrchestrator

Run cells in order to execute a full simulation.

In [ ]:
import json, os, re, uuid, random, math
from datetime import datetime
from typing import Dict, List, Optional
from pathlib import Path

import networkx as nx

REPO_ROOT = Path(os.getcwd()).parent
KB_DIR    = REPO_ROOT / 'vorq' / 'data' / 'kb'

print('Setup complete ✓')

## 1. Exposure Graph

In [ ]:
class ExposureGraph:
    """
    Weighted directed graph: countries → industries → companies → commodities.
    Models global supply-chain risk propagation.
    """

    def __init__(self, kb_dir: str = str(KB_DIR)):
        self.G = nx.DiGraph()
        self._build(kb_dir)

    def _build(self, kb_dir: str):
        with open(os.path.join(kb_dir, 'seed_knowledge_base.json')) as f:
            kb = json.load(f)
        with open(os.path.join(kb_dir, 'trade_dependencies.json')) as f:
            trades = json.load(f)

        for c   in kb['countries']:    self.G.add_node(c['id'],    type='country',   **c)
        for ind in kb['industries']:   self.G.add_node(ind['id'],  type='industry',  **ind)
        for co  in kb['companies']:    self.G.add_node(co['id'],   type='company',   **co)
        for cm  in kb['commodities']:  self.G.add_node(cm['id'],   type='commodity', **cm)

        for c in kb['countries']:
            for iid in c['key_industries']:
                if iid in self.G: self.G.add_edge(c['id'], iid, weight=0.7, relation='has_industry')
        for ind in kb['industries']:
            for cm_id in ind.get('commodities', []):
                if cm_id in self.G: self.G.add_edge(ind['id'], cm_id, weight=0.6, relation='uses_commodity')
        for co in kb['companies']:
            self.G.add_edge(co['id'], co['industry'], weight=0.8, relation='in_industry')
            self.G.add_edge(co['id'], co['country'],  weight=0.5, relation='headquartered_in')
            for dep in co.get('supply_dependencies', []):
                if dep in self.G: self.G.add_edge(co['id'], dep, weight=0.9, relation='depends_on')
        for e in trades['edges']:
            if e['from'] in self.G and e['to'] in self.G:
                self.G.add_edge(e['from'], e['to'], weight=e['weight'], relation='trades_with')

    def get_downstream(self, node_id, max_depth=3):
        if node_id not in self.G: return []
        visited, result, queue = set(), [], [(node_id, 0, 1.0, [node_id])]
        while queue:
            cur, depth, cw, path = queue.pop(0)
            if depth > max_depth or cur in visited: continue
            visited.add(cur)
            if cur != node_id:
                result.append({'id': cur, 'depth': depth, 'cumulative_weight': round(cw, 4),
                               'path': path.copy(), **dict(self.G.nodes[cur])})
            for nb in self.G.successors(cur):
                if nb not in visited:
                    ew = self.G[cur][nb].get('weight', 0.5)
                    queue.append((nb, depth+1, cw*ew, path+[nb]))
        result.sort(key=lambda x: x['cumulative_weight'], reverse=True)
        return result

    def get_all_paths(self, src, tgt, max_depth=5):
        try:    return list(nx.all_simple_paths(self.G, src, tgt, cutoff=max_depth))
        except: return []

    def get_path_weight(self, path):
        w = 1.0
        for i in range(len(path)-1):
            e = self.G.get_edge_data(path[i], path[i+1])
            w *= e.get('weight', 0.5) if e else 0.1
        return round(w, 4)

    def get_node_info(self, nid):
        return {'id': nid, **dict(self.G.nodes[nid])} if nid in self.G else None

    def get_neighbors(self, nid):
        if nid not in self.G: return []
        return [{'id': n, **dict(self.G.nodes[n]),
                 'edge_weight': self.G[nid][n].get('weight',0),
                 'relation': self.G[nid][n].get('relation','unknown')}
                for n in self.G.neighbors(nid)]

    def summary(self):
        return {
            'nodes': self.G.number_of_nodes(), 'edges': self.G.number_of_edges(),
            **{t: len([n for n,d in self.G.nodes(data=True) if d.get('type')==t])
               for t in ['country','industry','company','commodity']}
        }

    def to_json(self):
        return {
            'nodes': [{'id': n, **d} for n, d in self.G.nodes(data=True)],
            'edges': [{'source': u, 'target': v, **d} for u, v, d in self.G.edges(data=True)],
        }

graph = ExposureGraph()
print('ExposureGraph:', graph.summary())

## 2. Event Extractor

In [ ]:
# Run notebook 01 first (or re-define EntityResolver here)
import importlib.util, sys

# Inline the resolver so this notebook is self-contained
_resolver_src = str(REPO_ROOT / 'vorq' / 'data' / 'entity_resolver.py')
if Path(_resolver_src).exists():
    spec = importlib.util.spec_from_file_location('entity_resolver', _resolver_src)
    _mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(_mod)
    EntityResolver = _mod.EntityResolver
    print('EntityResolver loaded from .py ✓')
else:
    # Inline fallback (copy from notebook 01)
    print('entity_resolver.py not found — use notebook 01 definition')

In [ ]:
EVENT_TYPES = {
    'war':              {'severity_base': 0.95, 'keywords': ['war','conflict','military','invasion','attack','strike','bomb','missile']},
    'sanctions':        {'severity_base': 0.75, 'keywords': ['sanction','sanctions','embargo','ban','restrict','tariff','tariffs','trade war']},
    'pandemic':         {'severity_base': 0.85, 'keywords': ['pandemic','epidemic','outbreak','virus','covid','disease','lockdown','quarantine']},
    'natural_disaster': {'severity_base': 0.70, 'keywords': ['earthquake','tsunami','hurricane','typhoon','flood','drought','wildfire','volcano']},
    'supply_shock':     {'severity_base': 0.65, 'keywords': ['supply shock','shortage','disruption','blockade','supply chain','bottleneck']},
    'political_crisis': {'severity_base': 0.60, 'keywords': ['coup','revolution','protest','unrest','political crisis','regime change']},
    'economic_crisis':  {'severity_base': 0.70, 'keywords': ['recession','crash','default','bankruptcy','inflation','hyperinflation']},
    'cyberattack':      {'severity_base': 0.55, 'keywords': ['cyberattack','hack','breach','ransomware','cyber','malware']},
    'energy_crisis':    {'severity_base': 0.65, 'keywords': ['oil shock','energy crisis','pipeline','opec','oil supply','gas supply']},
    'trade_agreement':  {'severity_base': 0.30, 'keywords': ['trade deal','trade agreement','partnership','alliance','cooperation']},
}
SEVERITY_MODIFIERS = {
    'major':1.2,'massive':1.3,'minor':0.6,'small':0.5,'full-scale':1.4,'limited':0.7,
    'partial':0.8,'total':1.3,'severe':1.2,'catastrophic':1.5,'devastating':1.4,'global':1.3,
}
HORIZON_PATTERNS = [
    (r'(\d+)\s*year',  lambda m: int(m.group(1))*365),
    (r'(\d+)\s*month', lambda m: int(m.group(1))*30),
    (r'(\d+)\s*week',  lambda m: int(m.group(1))*7),
    (r'(\d+)\s*day',   lambda m: int(m.group(1))),
    (r'next\s+year',   lambda m: 365),
    (r'next\s+month',  lambda m: 30),
    (r'next\s+quarter',lambda m: 90),
]

class EventExtractor:
    """Rule-based NLP event extractor for VORQ scenarios."""

    def __init__(self):
        self.resolver = EntityResolver()

    def extract(self, scenario_text: str, horizon_days: Optional[int] = None) -> Dict:
        text_lower = scenario_text.lower().strip()
        event_type, type_confidence = self._detect_type(text_lower)
        entities = self.resolver.resolve_all(scenario_text)

        if not entities['industries'] and entities['countries']:
            inferred = set()
            for c in entities['countries']:
                for iid in c.get('key_industries', []): inferred.add(iid)
            for iid in inferred:
                ind = self.resolver.resolve_industry(iid)
                if ind: entities['industries'].append(ind)

        if not entities['companies'] and (entities['countries'] or entities['industries']):
            seen = set()
            for c in entities['countries']:
                for co in self.resolver.get_companies_by_country(c['id']): seen.add(co['id'])
            for ind in entities['industries']:
                for co in self.resolver.get_companies_by_industry(ind['id']): seen.add(co['id'])
            for cid in list(seen)[:10]:
                co = self.resolver._company_index.get(cid)
                if co: entities['companies'].append(co)

        severity = self._severity(text_lower, event_type)
        if horizon_days is None: horizon_days = self._horizon(text_lower)

        return {
            'event_id': str(uuid.uuid4())[:8],
            'type': event_type, 'severity': round(severity, 3),
            'entities': {
                'countries':  [{'id': c['id'], 'name': c['name']} for c in entities['countries']],
                'industries': [{'id': i['id'], 'label': i['label']} for i in entities['industries']],
                'companies':  [{'id': c['id'], 'name': c['name'], 'ticker': c.get('ticker','')} for c in entities['companies']],
            },
            'horizon_days': horizon_days,
            'confidence': round(type_confidence, 3),
            'parsed_at': datetime.utcnow().isoformat(),
            'source_text': scenario_text,
        }

    def _detect_type(self, text):
        best, score = 'supply_shock', 0.0
        for etype, cfg in EVENT_TYPES.items():
            s = sum(len(kw.split())*0.3 for kw in cfg['keywords'] if kw in text)
            if s > score: score, best = s, etype
        return best, min(0.95, 0.4 + score * 0.2)

    def _severity(self, text, etype):
        base = EVENT_TYPES.get(etype, {}).get('severity_base', 0.5)
        mod  = max((v for k, v in SEVERITY_MODIFIERS.items() if k in text), default=1.0)
        return min(1.0, base * mod)

    def _horizon(self, text):
        for pattern, fn in HORIZON_PATTERNS:
            m = re.search(pattern, text)
            if m: return fn(m)
        return 180

print('EventExtractor defined ✓')

## 3. Risk Scoring Engine (Monte Carlo)

In [ ]:
IMPACT_PROFILES = {
    'war':             {'direct_country_impact':(-0.35,-0.15),'neighbor_country_impact':(-0.15,-0.05),'industry_multipliers':{'defense':(0.10,0.30),'energy':(-0.25,0.15),'technology':(-0.30,-0.10),'manufacturing':(-0.25,-0.10),'shipping':(-0.20,-0.05),'agriculture':(-0.15,-0.05),'finance':(-0.15,-0.05),'healthcare':(-0.05,0.10),'automotive':(-0.20,-0.10),'mining':(-0.10,0.10)},'time_to_peak_days':(7,60),'recovery_months':(6,24)},
    'sanctions':       {'direct_country_impact':(-0.20,-0.08),'neighbor_country_impact':(-0.08,-0.02),'industry_multipliers':{'technology':(-0.25,-0.10),'energy':(-0.15,0.10),'manufacturing':(-0.15,-0.05),'finance':(-0.12,-0.03),'shipping':(-0.10,-0.03),'defense':(-0.05,0.15),'healthcare':(-0.05,0.05),'automotive':(-0.15,-0.05),'agriculture':(-0.08,-0.02),'mining':(-0.10,0.05)},'time_to_peak_days':(14,90),'recovery_months':(3,18)},
    'pandemic':        {'direct_country_impact':(-0.30,-0.10),'neighbor_country_impact':(-0.20,-0.08),'industry_multipliers':{'healthcare':(0.05,0.35),'technology':(-0.10,0.15),'shipping':(-0.25,-0.10),'manufacturing':(-0.25,-0.10),'automotive':(-0.25,-0.10),'energy':(-0.30,-0.10),'agriculture':(-0.10,0.05),'finance':(-0.20,-0.05),'defense':(-0.05,0.05),'mining':(-0.15,-0.05)},'time_to_peak_days':(30,120),'recovery_months':(6,36)},
    'supply_shock':    {'direct_country_impact':(-0.15,-0.05),'neighbor_country_impact':(-0.08,-0.02),'industry_multipliers':{'manufacturing':(-0.20,-0.08),'technology':(-0.15,-0.05),'automotive':(-0.15,-0.05),'shipping':(-0.10,0.10),'energy':(-0.10,0.10),'agriculture':(-0.10,-0.03),'healthcare':(-0.08,-0.02),'finance':(-0.05,-0.01),'defense':(-0.05,0.05),'mining':(-0.08,0.05)},'time_to_peak_days':(7,45),'recovery_months':(2,12)},
    'energy_crisis':   {'direct_country_impact':(-0.20,-0.08),'neighbor_country_impact':(-0.10,-0.03),'industry_multipliers':{'energy':(-0.15,0.25),'manufacturing':(-0.20,-0.08),'shipping':(-0.15,-0.05),'automotive':(-0.15,-0.05),'technology':(-0.08,-0.02),'agriculture':(-0.10,-0.03),'finance':(-0.08,-0.02),'healthcare':(-0.05,-0.01),'defense':(0.02,0.10),'mining':(-0.05,0.08)},'time_to_peak_days':(7,60),'recovery_months':(3,18)},
}
_DEFAULT_PROFILE = {'direct_country_impact':(-0.15,-0.05),'neighbor_country_impact':(-0.05,-0.01),'industry_multipliers':{ind:(-0.10,-0.02) for ind in ['technology','finance','healthcare','energy','defense','automotive','manufacturing','agriculture','mining','shipping']},'time_to_peak_days':(14,60),'recovery_months':(3,12)}

class RiskScoringEngine:
    def __init__(self, mc_iterations=1000):
        self.graph = ExposureGraph()
        self.mc_iterations = mc_iterations

    def simulate(self, event: Dict) -> Dict:
        etype    = event.get('type', 'supply_shock')
        severity = event.get('severity', 0.5)
        horizon  = event.get('horizon_days', 180)
        profile  = IMPACT_PROFILES.get(etype, _DEFAULT_PROFILE)
        mults    = profile.get('industry_multipliers', {})

        # ── Country impacts
        country_impacts = {}
        for c in event['entities'].get('countries', []):
            lo, hi = profile['direct_country_impact']
            country_impacts[c['id']] = {'id': c['id'], 'name': c['name'], 'impact_pct': round((lo+hi)/2*severity*100, 2), 'type': 'direct'}
        for cid in list(country_impacts):
            for node in self.graph.get_downstream(cid, max_depth=1):
                if node.get('type') == 'country' and node['id'] not in country_impacts:
                    lo, hi = profile['neighbor_country_impact']
                    country_impacts[node['id']] = {'id': node['id'], 'name': node.get('name', node['id']), 'impact_pct': round((lo+hi)/2*severity*node['cumulative_weight']*100, 2), 'type': 'indirect'}

        # ── Industry impacts
        industry_impacts = {}
        for ind in event['entities'].get('industries', []):
            lo, hi = mults.get(ind['id'], (-0.10, -0.02))
            base = (lo+hi)/2*severity
            industry_impacts[ind['id']] = {'id': ind['id'], 'label': ind['label'], 'impact_pct': round(base*100, 2), 'direction': 'positive' if base > 0 else 'negative'}

        # ── Company impacts (Monte Carlo)
        company_impacts = []
        for comp in event['entities'].get('companies', []):
            ni = self.graph.get_node_info(comp['id'])
            ind = ni.get('industry') if ni else None
            lo, hi = mults.get(ind, (-0.10,-0.02)) if ind else (-0.10,-0.02)
            mc = self._mc(lo, hi, severity, comp['id'])
            paths = self._paths(comp['id'], event)
            company_impacts.append({'id': comp['id'], 'name': comp['name'], 'ticker': comp.get('ticker',''), 'industry': ind or 'unknown', 'expected_impact_pct': mc['expected_pct'], 'impact_range': mc['range'], 'p_impact_gt_10pct': mc['p_gt_10'], 'p_impact_gt_20pct': mc['p_gt_20'], 'confidence_interval_95': mc['ci_95'], 'time_to_impact_days': self._time(profile, severity), 'risk_score': mc['risk_score'], 'direction': 'positive' if mc['expected_pct'] > 0 else 'negative', 'contributing_paths': paths[:3]})
        company_impacts.sort(key=lambda x: abs(x['expected_impact_pct']), reverse=True)

        # ── Timeline
        lo_t, hi_t = profile['time_to_peak_days']
        lo_r, hi_r = profile['recovery_months']
        peak     = int((lo_t+hi_t)/2*severity)
        recovery = int((lo_r+hi_r)/2*severity)

        avg_abs  = sum(abs(c['expected_impact_pct']) for c in company_impacts)/len(company_impacts) if company_impacts else 5.0
        overall  = min(100, avg_abs * severity * 10)

        return {'event': {'type': etype, 'severity': severity, 'horizon_days': horizon, 'confidence': event.get('confidence', 0.5)}, 'overall_risk_score': round(overall, 1), 'risk_level': self._level(overall), 'country_impacts': list(country_impacts.values()), 'industry_impacts': list(industry_impacts.values()), 'company_impacts': company_impacts, 'timeline': {'time_to_peak_days': peak, 'recovery_months': recovery, 'horizon_days': horizon}, 'simulation_config': {'monte_carlo_iterations': self.mc_iterations, 'engine_version': '1.0'}}

    def _mc(self, lo, hi, severity, comp_id):
        results = [random.uniform(lo,hi)*severity*random.uniform(0.7,1.3)*(1.0+len(self.graph.get_neighbors(comp_id))*0.02)*100 for _ in range(self.mc_iterations)]
        results.sort(); n = len(results)
        exp = sum(results)/n
        return {'expected_pct': round(exp,2), 'range':[round(results[0],2),round(results[-1],2)], 'p_gt_10': round(sum(1 for r in results if abs(r)>10)/n,3), 'p_gt_20': round(sum(1 for r in results if abs(r)>20)/n,3), 'ci_95': [round(results[int(n*.025)],2), round(results[int(n*.975)],2)], 'risk_score': round(min(100,abs(exp)*5+sum(1 for r in results if abs(r)>10)/n*30),1)}

    def _paths(self, comp_id, event):
        paths = []
        for c in event['entities'].get('countries',[]):
            for p in self.graph.get_all_paths(c['id'], comp_id, max_depth=4)[:3]:
                paths.append({'path': p, 'weight': self.graph.get_path_weight(p), 'description': ' → '.join(p)})
        paths.sort(key=lambda x: x['weight'], reverse=True)
        return paths

    def _time(self, profile, severity):
        lo, hi = profile['time_to_peak_days']
        return int(lo + (hi-lo)*(1-severity))

    def _level(self, score):
        if score >= 80: return 'CRITICAL'
        if score >= 60: return 'HIGH'
        if score >= 40: return 'ELEVATED'
        if score >= 20: return 'MODERATE'
        return 'LOW'

print('RiskScoringEngine defined ✓')

## 4. Explainability Engine

In [ ]:
MITIGATIONS = {
    'war':           ['Diversify supply chains away from conflict zones','Increase inventory buffers for critical components (30-90 day stock)','Hedge currency exposure for affected regions','Identify alternative suppliers in neutral countries','Review force majeure clauses in existing contracts'],
    'sanctions':     ['Audit supply chain for sanctioned entity exposure','Pre-qualify alternative suppliers in non-sanctioned jurisdictions','Ensure compliance with new export control regulations','Consider nearshoring critical production','Engage legal counsel for sanctions compliance review'],
    'pandemic':      ['Establish remote work protocols for critical operations','Diversify manufacturing across multiple geographies','Build strategic reserves of essential inputs','Implement digital supply chain visibility tools','Review business continuity and disaster recovery plans'],
    'supply_shock':  ['Map tier-2 and tier-3 suppliers for visibility','Negotiate long-term supply agreements with volume guarantees','Invest in substitute materials R&D','Build strategic buffer stock (60-90 days)','Establish dual-sourcing for critical components'],
    'energy_crisis': ['Evaluate energy hedging strategies (futures/options)','Invest in energy efficiency improvements','Explore renewable energy alternatives','Negotiate long-term energy supply contracts','Review backup power generation capabilities'],
    'economic_crisis':['Strengthen balance sheet — increase cash reserves','Reduce exposure to high-yield / speculative assets','Lock in fixed-rate financing where possible','Stress-test portfolio under severe recession scenarios','Identify counter-cyclical investment opportunities'],
    'cyberattack':   ['Conduct immediate security audit of critical systems','Implement zero-trust architecture','Establish incident response team and playbooks','Review cyber insurance coverage and limits','Segment networks to contain potential breaches'],
}

class ExplainabilityEngine:
    def __init__(self):
        self.graph = ExposureGraph()

    def explain(self, event: Dict, sim: Dict) -> Dict:
        etype = event.get('type', 'supply_shock')
        return {
            'summary': self._summary(event, sim),
            'causal_chains': self._chains(event, sim),
            'sector_analysis': self._sectors(sim),
            'sensitivity_breakdown': self._sensitivity(event, sim),
            'mitigation_suggestions': MITIGATIONS.get(etype, ['Monitor situation closely','Review risk exposure','Prepare contingency plans']),
            'key_risks': self._risks(sim),
            'confidence_assessment': self._confidence(event, sim),
        }

    def _summary(self, event, sim):
        et    = event['type'].replace('_',' ').title()
        sev_l = 'Severe' if event['severity']>0.8 else 'Significant' if event['severity']>0.5 else 'Moderate'
        ctrs  = ' and '.join(c['name'] for c in event['entities'].get('countries',[]))
        rl, rs = sim.get('risk_level','MODERATE'), sim.get('overall_risk_score',0)
        nc, ni = len(sim.get('company_impacts',[])), len(sim.get('industry_impacts',[]))
        worst  = max(sim['company_impacts'], key=lambda x: abs(x['expected_impact_pct'])) if sim.get('company_impacts') else None
        tl     = sim.get('timeline', {})
        parts  = [f'**{et}** scenario involving {ctrs or "Global"}.', f'Assessment: **{sev_l}** event with **{rl}** risk level (score: {rs}/100).', f'Impact spans **{ni} industries** and **{nc} companies** tracked.']
        if worst: parts.append(f'Highest single-entity impact: {worst["name"]} ({worst["expected_impact_pct"]:+.1f}%).')
        if tl: parts.append(f'Peak impact expected in **{tl.get("time_to_peak_days","N/A")} days**, recovery estimated at **{tl.get("recovery_months","N/A")} months**.')
        return ' '.join(parts)

    def _chains(self, event, sim):
        chains = []
        for comp in sim.get('company_impacts',[])[:5]:
            for pi in comp.get('contributing_paths',[])[:2]:
                p = pi.get('path',[])
                if len(p) >= 2:
                    chains.append({'company': comp['name'], 'path': p, 'weight': pi.get('weight',0), 'explanation': f'Risk propagates from {p[0]} → {p[-1]} (weight: {pi.get("weight",0):.3f})'})
        chains.sort(key=lambda x: x['weight'], reverse=True)
        return chains[:10]

    def _sectors(self, sim):
        out = []
        for ind in sim.get('industry_impacts',[]):
            imp = ind['impact_pct']
            out.append({'sector': ind['label'], 'impact_pct': imp, 'direction': ind['direction'], 'severity': 'HIGH' if abs(imp)>15 else 'MEDIUM' if abs(imp)>8 else 'LOW', 'emoji': '📉' if imp<0 else '📈'})
        out.sort(key=lambda x: abs(x['impact_pct']), reverse=True)
        return out

    def _sensitivity(self, event, sim):
        sev = event.get('severity', 0.5)
        nc  = len(event['entities'].get('countries',[]))
        ni  = len(event['entities'].get('industries',[]))
        return {'severity_contribution': round(sev*40,1), 'geographic_spread': round(min(30,nc*15),1), 'industry_breadth': round(min(20,ni*4),1), 'supply_chain_depth': round(10*sev,1), 'interpretation': f'Event severity accounts for {sev*40:.0f}%. Geographic spread ({nc} countries) adds {min(30,nc*15):.0f}%. Industry breadth ({ni} sectors) contributes {min(20,ni*4):.0f}%.'}

    def _risks(self, sim):
        risks = []
        for co in sim.get('company_impacts',[])[:3]:
            if abs(co['expected_impact_pct']) > 15:
                risks.append(f'⚠️ {co["name"]} faces {co["expected_impact_pct"]:+.1f}% impact with {co["p_impact_gt_10pct"]*100:.0f}% probability of >10% move')
        for co in sim.get('company_impacts',[]):
            if len(co.get('contributing_paths',[])) >= 2:
                risks.append(f'🔗 {co["name"]} exposed through {len(co["contributing_paths"])} supply chain paths')
                break
        severe = [i for i in sim.get('industry_impacts',[]) if abs(i['impact_pct'])>12]
        if severe: risks.append(f'🏭 Severe sector disruption in: {", ".join(i["label"] for i in severe[:3])}')
        if not risks: risks.append('ℹ️ No critical risk thresholds breached — monitor situation')
        return risks

    def _confidence(self, event, sim):
        ec   = event.get('confidence', 0.5)
        paths = sum(len(c.get('contributing_paths',[])) for c in sim.get('company_impacts',[]))
        dr   = min(1.0, paths/10)
        ov   = ec*.4 + dr*.3 + 0.3
        return {'overall': round(ov,2), 'event_parsing': round(ec,2), 'data_coverage': round(dr,2), 'model_confidence': 0.7}

print('ExplainabilityEngine defined ✓')

## 5. Orchestrator + Live Simulation Test

In [ ]:
class SimulationOrchestrator:
    def __init__(self, mc_iterations=1000):
        self.extractor = EventExtractor()
        self.scorer    = RiskScoringEngine(mc_iterations=mc_iterations)
        self.explainer = ExplainabilityEngine()

    def run(self, scenario_text, horizon_days=None, severity_override=None, mc_iterations=None):
        if mc_iterations: self.scorer.mc_iterations = mc_iterations
        event = self.extractor.extract(scenario_text, horizon_days)
        if severity_override is not None:
            event['severity'] = max(0.0, min(1.0, severity_override))
        sim  = self.scorer.simulate(event)
        expl = self.explainer.explain(event, sim)
        return {'status': 'success', 'event': event, 'simulation': sim, 'explanation': expl}

    def quick_score(self, scenario_text):
        event = self.extractor.extract(scenario_text)
        sim   = self.scorer.simulate(event)
        return {'event_type': event['type'], 'severity': event['severity'], 'overall_risk_score': sim['overall_risk_score'], 'risk_level': sim['risk_level'], 'top_impacts': [{'name': c['name'], 'impact': c['expected_impact_pct']} for c in sim['company_impacts'][:5]], 'time_to_peak': sim['timeline']['time_to_peak_days']}

print('SimulationOrchestrator defined ✓')

In [ ]:
# ── Live simulation test ──────────────────────────────────────────────────────
import time

orchestrator = SimulationOrchestrator(mc_iterations=500)  # 500 for speed in notebook

scenarios = [
    'War between India and China in 1 year',
    'US sanctions on semiconductor exports to China',
    'Major oil supply shock in the Middle East',
]

for s in scenarios:
    t0 = time.time()
    result = orchestrator.run(s)
    elapsed = round(time.time()-t0, 2)
    sim = result['simulation']
    print(f'\n"{s}"')
    print(f'  Risk: {sim["overall_risk_score"]} ({sim["risk_level"]})  |  {elapsed}s')
    print(f'  Companies: {[c["name"] for c in sim["company_impacts"][:3]]}')
    print(f'  Peak in {sim["timeline"]["time_to_peak_days"]}d, recovery ~{sim["timeline"]["recovery_months"]}mo')